# LangGraph 三智能体协作 · Researcher + Reviewer + Writer

**任务**：给定一个主题（默认 *2025 LLM Agent 进展*），生成一段 200-300 字的小综述。

三角色：
- **Researcher**：根据主题列 5-7 个要点（用一个迷你检索工具拿证据）。
- **Reviewer**：审视要点，提出补充/修改意见。
- **Writer**：基于要点 + 评审意见写最终段落。

我们用 **LangGraph** 显式编排（节点 + 边 + 状态），便于调试与扩展。

In [ ]:
import os, sys
sys.path.append(os.path.abspath('../..'))
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, END
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model='claude-sonnet-4-5', temperature=0)
llm_lo = ChatAnthropic(model='claude-sonnet-4-5', temperature=0.3)

## 1. 状态定义

LangGraph 状态用 TypedDict；每轮节点返回一个 dict 来 *合并*。

In [ ]:
class S(TypedDict, total=False):
    topic: str
    bullets: str        # researcher 输出
    review: str         # reviewer 输出
    revisions: int      # 已迭代次数
    article: str        # writer 输出
    approved: bool      # reviewer 是否放行


## 2. 三个 agent 节点函数

In [ ]:
RESEARCHER_SYS = (
    '你是研究员。基于主题，列出 5-7 个核心要点（每条 1-2 句），覆盖关键论文/项目/趋势。'
    '若有 reviewer 反馈，请依据反馈修订。请直接输出要点，不要前后客套。'
)
REVIEWER_SYS = (
    '你是评审。判断研究员的要点是否充分（覆盖度、准确性、是否聚焦 2024-2026 进展）。'
    '如果通过，第一行只写 APPROVED；否则第一行写 NEEDS_REVISION，后面写具体改进意见（不超过 5 条）。'
)
WRITER_SYS = (
    '你是写作助手。把通过评审的要点串成一段 200-300 字、逻辑清晰的中文综述，'
    '保持客观语调，不要列表，不要 markdown 标题，只输出正文段落。'
)

def node_research(state: S) -> S:
    fb = state.get('review', '')
    user = f'主题：{state["topic"]}'
    if fb and not state.get('approved'):
        user += f'\n\nReviewer 反馈：\n{fb}\n请据此修订要点。'
    out = llm_lo.invoke([
        ('system', RESEARCHER_SYS),
        ('user', user),
    ])
    return {'bullets': out.content, 'revisions': state.get('revisions', 0) + 1}

def node_review(state: S) -> S:
    out = llm.invoke([
        ('system', REVIEWER_SYS),
        ('user', f'要点如下：\n{state["bullets"]}'),
    ])
    text = out.content
    return {'review': text, 'approved': text.strip().splitlines()[0].strip().upper() == 'APPROVED'}

def node_write(state: S) -> S:
    out = llm.invoke([
        ('system', WRITER_SYS),
        ('user', f'要点：\n{state["bullets"]}'),
    ])
    return {'article': out.content}

## 3. 图结构

researcher → reviewer →（approved? Y → writer / N → researcher，最多 3 次）→ END

In [ ]:
g = StateGraph(S)
g.add_node('research', node_research)
g.add_node('review', node_review)
g.add_node('write', node_write)
g.set_entry_point('research')
g.add_edge('research', 'review')

def route(state: S):
    if state.get('approved'):
        return 'write'
    if state.get('revisions', 0) >= 3:
        return 'write'  # 强制收敛，避免死循环
    return 'research'

g.add_conditional_edges('review', route, {'research': 'research', 'write': 'write'})
g.add_edge('write', END)
graph = g.compile()

In [ ]:
result = graph.invoke({'topic': '2025 LLM Agent 进展（覆盖框架、评测、RL 训练、计算机使用）'})
print('=== 要点（共修订', result.get('revisions'), '次）===')
print(result['bullets'])
print()
print('=== 评审意见 ===')
print(result['review'])
print()
print('=== 综述 ===')
print(result['article'])

## 4. 思考

- 用 *显式 graph + state* 来组织多 agent，比纯对话框架（早期 AutoGen）更可控。
- *条件边 + 修订计数* 是控制收敛的关键，否则容易在「研究员-评审」之间死循环。
- 三 agent 比单 agent 多花约 3-5 倍 token；要在「重要决策」节点才上多 agent。

## 进阶练习

1. 给 Researcher 接入真实 web 搜索（tavily / serper），把 bullets 上加引用。
2. 加第四个 *Critic* agent 用 Multi-Agent Debate，看输出质量是否进一步提升。
3. 用 LangSmith 追踪每节点的耗时与 token，画出火焰图。